# S03 · 02 — Tracking con MLflow

El notebook anterior dejó cinco preguntas sin respuesta. Aquí se responden todas,
y aparece una sexta que casi ningún tutorial menciona: **¿el artefacto que
guardaste se puede servir?**

## Qué vas a hacer

1. Registrar un entrenamiento completo: tags, params, métricas y artifacts.
2. Comparar corridas con `mlflow.search_runs()` en lugar de con la vista.
3. Usar `mlflow.autolog()` y descubrir sus límites.
4. Loguear el modelo con `signature` e `input_example`, y **verlo fallar** cuando
   la petición no cumple el contrato.
5. Entender por qué el `serialization_format` por defecto cambió a `skops`.

## Antes de empezar

En una terminal aparte, desde la raíz del repositorio:

```bash
make mlflow
# equivale a:
# uv run mlflow server --backend-store-uri sqlite:///mlflow.db \
#   --default-artifact-root ./mlartifacts --host 127.0.0.1 --port 5001
```

Abre <http://127.0.0.1:5001> y confirma que la UI carga.

> **Por qué 5001, una sola vez para todo el curso:** en macOS, AirPlay Receiver
> ocupa el puerto por defecto de `mlflow server` y responde un HTTP 403 que no
> dice nada útil. El valor vive en `taxi.config.MLFLOW_PORT` y lo leen los
> scripts, los notebooks, el `docker-compose.yml` y el `.env.example`. Antes del
> rediseño los scripts y los notebooks usaban puertos distintos, y el estudiante
> acababa con el servidor en uno y el cliente en el otro.

## 1. Anatomía de MLflow: tres piezas, no una

"MLflow" es el nombre de cuatro cosas que se pueden desplegar por separado. La
confusión más común de la sesión sale de tratarlas como una sola.

```mermaid
flowchart LR
    CLI["Tu codigo<br/>mlflow.log_metric(...)"] -->|HTTP| TS["Tracking server<br/>:5001"]
    TS --> BS[("Backend store<br/>metadata: runs, params,<br/>metrics, versiones")]
    TS --> AS[("Artifact store<br/>archivos: modelo, plots,<br/>tablas")]
    TS --- REG["Model Registry<br/><i>necesita backend de BD</i>"]
```

| Pieza | Qué guarda | En este curso | En producción |
|---|---|---|---|
| Tracking server | nada: es la API | `mlflow server` en `:5001` | contenedor detrás de un proxy con auth |
| Backend store | metadata (runs, params, métricas, versiones del registry) | SQLite (`mlflow.db`) | Postgres gestionado (RDS) |
| Artifact store | archivos (modelo, figuras, tablas) | carpeta local (`./mlartifacts`) | S3 / MinIO |
| Model Registry | nombres, versiones, aliases y tags | habilitado por el backend SQLite | el mismo, con permisos |

**La regla que hay que recordar:** el Model Registry **necesita** un backend de
base de datos. Con un *file store* (`file://mlruns`) no existe. Es exactamente lo
que separa al escenario 1 del escenario 2 en [`../scenarios/`](../scenarios/).

In [ ]:
import matplotlib.pyplot as plt
import mlflow
import pandas as pd
from mlflow.models import infer_signature
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error

from taxi import config
from taxi.features import contract as fc
from taxi.models import train

mlflow.set_tracking_uri(config.MLFLOW_TRACKING_URI)
print("tracking URI:", mlflow.get_tracking_uri())
print("experimentos existentes:", [e.name for e in mlflow.search_experiments()])

### Convención de nombres de experimento

Los nombres viven en `taxi.config.EXPERIMENTOS` con el formato `s0X-proposito`.
No es cosmética: antes había cuatro nombres distintos (`nyc-taxi-experiment`,
`nyc-first-experiment`, `nyc-taxi-baseline`, `nyc-taxi-tracking`) para dos
problemas, y las corridas del mismo modelo quedaban repartidas entre ellos, así
que no se podían comparar.

In [ ]:
for clave, nombre in config.EXPERIMENTOS.items():
    print(f"{clave:12s} -> {nombre}")

df_train = train.cargar_train()
df_valid = train.cargar_valid()
y_valid = df_valid[fc.TARGET_REGRESION].to_numpy(dtype=float)
print(df_train.shape, df_valid.shape)

## 2. Las cuatro cosas que se registran

| Tipo | Para qué sirve | Se puede cambiar después |
|---|---|---|
| **params** | reproducir la corrida | no: son inmutables en el run |
| **metrics** | comparar corridas (y admiten `step` para curvas) | se pueden añadir puntos |
| **tags** | filtrar y buscar (`tags.tipo = 'baseline'`) | sí |
| **artifacts** | todo lo que es un archivo: modelo, figuras, tablas | se añaden, no se editan |

Una métrica responde "¿qué tan bueno?"; un tag responde "¿de qué corrida
estamos hablando?". Confundirlos es lo que produce experimentos con 40 runs y
ninguna forma de agrupar.

In [ ]:
pipeline = train.pipeline_random_forest(max_depth=10, n_estimators=25)
train.ajustar(pipeline, df_train, df_valid)
y_pred = pipeline.predict(df_valid)

mlflow.set_experiment(config.EXPERIMENTOS["baseline"])

with mlflow.start_run(run_name="rf-manual-depth10") as run:
    # --- Tags: metadata para filtrar despues ---
    mlflow.set_tags(
        {
            "tipo": "baseline",
            "model_family": "random_forest",
            "caso": "nyc-green-taxi",
            # Que datos: sin esto, dos runs con el mismo RMSE son
            # indistinguibles y la comparacion no significa nada.
            "particiones_train": ",".join(p.etiqueta for p in config.PARTICIONES_TRAIN),
            "particion_valid": config.PARTICION_VALID.etiqueta,
            "features": ",".join(fc.FEATURES),
            "holdout_evaluado": "no",
        }
    )

    # --- Params ---
    mlflow.log_params(
        {
            "max_depth": 10,
            "n_estimators": 25,
            "semilla": config.SEMILLA,
            "filas_train": len(df_train),
            "filas_valid": len(df_valid),
            "n_features_vectorizadas": len(
                pipeline.named_steps["vectorizador"].get_feature_names_out()
            ),
        }
    )

    # --- Metricas ---
    mlflow.log_metrics(
        {
            "valid_rmse": float(root_mean_squared_error(y_valid, y_pred)),
            "valid_mae": float(mean_absolute_error(y_valid, y_pred)),
            "valid_r2": float(r2_score(y_valid, y_pred)),
        }
    )

    # --- Artifacts ---
    # log_figure y log_table escriben DIRECTO al artifact store. La version
    # anterior de este notebook hacia fig.savefig("residuals.png") y
    # log_artifact("residuals.png"): dejaba basura en el repositorio y el
    # artefacto dependia del directorio desde el que se abrio el notebook.
    fig, eje = plt.subplots(figsize=(8, 4))
    eje.scatter(y_pred, y_valid - y_pred, s=5, alpha=0.2)
    eje.axhline(0.0, color="red", linewidth=1)
    eje.set_xlabel("duracion predicha (min)")
    eje.set_ylabel("residual = real - predicha (min)")
    eje.set_title("Residuales en validacion")
    fig.tight_layout()
    mlflow.log_figure(fig, "graficos/residuales_valid.png")
    plt.close(fig)

    mlflow.log_table(
        pd.DataFrame({"y_true": y_valid[:500], "y_pred": y_pred[:500]}),
        "tablas/predicciones.json",
    )

    run_id_manual = run.info.run_id

print("run_id:", run_id_manual)
print("Abre la UI y busca este run en el experimento", config.EXPERIMENTOS["baseline"])

### Lo que acabas de ganar

Vuelve a las cinco preguntas del notebook 01 y respóndelas en la UI. Las cuatro
primeras ya tienen respuesta. La quinta —"¿se puede repetir?"— tiene una
respuesta *parcial*: sabes los parámetros y los datos, pero todavía no sabes con
qué versión del código, y el modelo aún no está guardado. Las dos cosas se
arreglan en este mismo notebook.

## 3. Comparar por código, no por vista

La UI sirve para mirar; `search_runs` sirve para **decidir**, porque devuelve un
DataFrame y eso se puede filtrar, ordenar y pegar en un informe.

In [ ]:
runs = mlflow.search_runs(
    experiment_names=[config.EXPERIMENTOS["baseline"]],
    order_by=["metrics.valid_rmse ASC"],
    max_results=10,
)

columnas = [c for c in ["run_id", "metrics.valid_rmse", "metrics.valid_mae",
                        "params.max_depth", "tags.model_family"] if c in runs.columns]
runs[columnas]

Filtrar es lo que hace útiles los tags. La sintaxis es SQL-ish:

```python
mlflow.search_runs(
    experiment_names=[config.EXPERIMENTOS["baseline"]],
    filter_string="tags.model_family = 'random_forest' and metrics.valid_rmse < 6",
)
```

## 4. `mlflow.autolog()` y sus límites

`autolog()` instrumenta la librería de ML por debajo: parchea `fit` y registra
lo que puede inferir. Ahorra código y es la mejor forma de empezar.

Lo que **no** puede saber, porque no está en la llamada a `fit`:

| Autolog registra | Autolog NO registra |
|---|---|
| hiperparámetros del estimador | qué particiones de datos usaste |
| métricas de entrenamiento | tu métrica de negocio |
| el modelo, con `signature` inferida | métricas por subgrupo |
| algunos artifacts del framework | por qué corriste este experimento |

Regla práctica: `autolog()` para los params y el modelo, logging manual para
**tags de datos y métricas de decisión**. Y un aviso: con `autolog()` activo, un
`GridSearchCV` puede generar decenas de runs hijos y ensuciar el experimento.

In [ ]:
mlflow.set_experiment(config.EXPERIMENTOS["comparacion"])
mlflow.sklearn.autolog()

with mlflow.start_run(run_name="rf-autolog") as run:
    # Lo unico que agregamos a mano es lo que autolog no puede inferir.
    mlflow.set_tags(
        {
            "tipo": "comparacion",
            "particiones_train": ",".join(p.etiqueta for p in config.PARTICIONES_TRAIN),
        }
    )
    pipeline_auto = train.pipeline_random_forest(max_depth=8, n_estimators=25)
    train.ajustar(pipeline_auto, df_train, df_valid)
    run_id_autolog = run.info.run_id

# Desactivar autolog al terminar: si queda activo, cada fit posterior crea un run
# y las celdas siguientes ensucian el experimento sin que se note.
mlflow.sklearn.autolog(disable=True)

datos = mlflow.get_run(run_id_autolog).data
print("params capturados:", len(datos.params))
print("metricas capturadas:", sorted(datos.metrics)[:8])

## 5. `signature` e `input_example`: el contrato del modelo

Un modelo sin `signature` es una función que acepta cualquier cosa y falla en
producción. La `signature` declara las columnas y sus tipos; MLflow los **exige**
al servir (*schema enforcement*).

Aquí está la trampa que solo aparece al servir, no al entrenar. El dataframe
procesado guarda `hora_pickup` como `int16` para ahorrar memoria. Si el
`input_example` se pasa tal cual, la firma declara la columna como `int32` y
MLflow **rechaza** una petición cuya columna llegue como `int64` —que es lo que
produce cualquier cliente normal, incluido pandas por defecto— con el mensaje
`Can not safely convert int64 to int32`.

**La regla:** una firma es un contrato con consumidores que no controlas. Declara
el tipo **más permisivo** que el modelo acepta, no el más compacto con el que se
entrenó. MLflow permite ensanchar (`int16` → `int64`) pero nunca estrechar.

Eso es lo que hace `train._ejemplo_de_entrada`: lee su docstring, el guion bajo
dice "detalle interno del paquete", pero el problema que resuelve es contenido de
esta clase.

In [ ]:
ejemplo = train._ejemplo_de_entrada(df_valid)
print(ejemplo.dtypes)

firma = infer_signature(ejemplo, pipeline.predict(ejemplo))
print(firma)

### Loguear el modelo

Tres cosas que hay que mirar en la llamada:

- `name="modelo"` y **no** `artifact_path=`: el segundo está deprecado en los
  flavors de MLflow 3.
- `skops_trusted_types`: la lista de tipos que no son de scikit-learn. Ver la
  sección 7.
- se loguea el **Pipeline completo**, que ya contiene el `DictVectorizer`. Un
  artefacto, una versión, un hash. El repo anterior guardaba `preprocessor.b` y
  `model.ubj` por separado y los copiaba a mano entre módulos: cuando se
  desincronizaban, el modelo predecía sobre features mal codificadas y **nada
  fallaba**.

In [ ]:
mlflow.set_experiment(config.EXPERIMENTOS["baseline"])

with mlflow.start_run(run_id=run_id_manual):
    info = mlflow.sklearn.log_model(
        sk_model=pipeline,
        name="modelo",
        signature=firma,
        input_example=ejemplo,
        skops_trusted_types=["taxi.models.train.ADiccionarios"],
    )

print("model_uri:", info.model_uri)

### Ahora véelo fallar

Tres peticiones al mismo modelo cargado como `pyfunc`: una correcta y dos que
rompen el contrato. Esto es *train/serve skew* detectado **antes** de producción,
que es el único momento en que sale barato.

In [ ]:
# El mensaje de enforcement de MLflow trae el dataframe completo y el schema
# antes de decir que paso. Lo que importa viene despues de "Error:".
def motivo(error: Exception) -> str:
    texto = " ".join(str(error).splitlines())
    corte = texto.find("Error:")
    return texto[corte:][:200] if corte >= 0 else texto[:200]


modelo = mlflow.pyfunc.load_model(info.model_uri)
print("1) tipos correctos ->", modelo.predict(ejemplo)[:3].round(3))

# 2) La hora llega como float (lo que produce cualquier JSON con 12.0)
peticion_float = ejemplo.copy()
peticion_float["hora_pickup"] = peticion_float["hora_pickup"].astype("float64")
try:
    modelo.predict(peticion_float)
    print("2) float donde se espera entero -> NO fallo (revisa la firma)")
except Exception as error:
    print("2) float donde se espera entero ->", type(error).__name__, "|", motivo(error))

# 3) Falta una columna del contrato
try:
    modelo.predict(ejemplo.drop(columns=["trip_distance"]))
    print("3) falta una feature -> NO fallo")
except Exception as error:
    print("3) falta una feature ->", type(error).__name__, "|", motivo(error))

Sin `signature`, el caso 3 no habría fallado: `DictVectorizer` ignora las claves
que no vio en `fit`, así que la petición habría producido una predicción
**silenciosamente peor**. Un error visible cuesta un ticket; un error silencioso
cuesta un trimestre de decisiones tomadas con predicciones malas.

## 6. `mlflow.models.evaluate`: evaluación como artifact

Calcular métricas a mano está bien. `mlflow.models.evaluate` las calcula y además
las **deja pegadas al modelo** en el run, junto con artifacts de diagnóstico.

Dos avisos sobre nombres, porque son una fuente real de confusión:

- El nombre canónico es **`mlflow.models.evaluate`**. `mlflow.evaluate` es un
  alias histórico; no lo uses en material nuevo.
- **`mlflow.genai.evaluate` es otra cosa**: es la API de evaluación de LLM (S08),
  con *scorers* y *judges*. **No son interoperables**: no comparten firma, ni
  tipos de dataset, ni métricas. Elegir la equivocada produce un error de tipos,
  no un resultado malo.

In [ ]:
datos_eval = df_valid[fc.FEATURES + [fc.TARGET_REGRESION]].head(2000)

with mlflow.start_run(run_id=run_id_manual):
    resultado = mlflow.models.evaluate(
        info.model_uri,
        data=datos_eval,
        targets=fc.TARGET_REGRESION,
        model_type="regressor",
    )

interesantes = ["root_mean_squared_error", "mean_absolute_error", "r2_score", "example_count"]
print({k: round(float(v), 4) for k, v in resultado.metrics.items() if k in interesantes})

## 7. `serialization_format`: skops es el nuevo default, y es una decisión de seguridad

En MLflow 3, `mlflow.sklearn.log_model` usa `serialization_format='skops'` por
defecto (antes era `cloudpickle`). Verificado contra **mlflow 3.15.1**.

| | `skops` (default) | `cloudpickle` |
|---|---|---|
| Al cargar | reconstruye solo tipos de una *allowlist* | **ejecuta el código** que venga en el archivo |
| Tipos no-sklearn | hay que declararlos en `skops_trusted_types` | funcionan sin declarar nada |
| Si falta un tipo | falla ruidosamente al loguear | no falla |
| Riesgo | acotado | ejecución arbitraria de código al deserializar |

Por qué importa de verdad: un artefacto de modelo es un archivo que **baja de un
bucket** y se carga en un proceso de producción. Con `cloudpickle`, cargar es
ejecutar.

Lo verificado en este repositorio (mlflow 3.15.1):

- Con un `Pipeline` que contiene `ADiccionarios`, `log_model` **falla** sin
  declarar `skops_trusted_types=["taxi.models.train.ADiccionarios"]`:
  *"The saved sklearn model references untrusted types"*.
- Si además hay un `XGBRegressor` dentro, hay que declarar tres:
  `ADiccionarios`, `xgboost.core.Booster` y `xgboost.sklearn.XGBRegressor`. Con
  los tres declarados, skops funciona.
- `src/taxi/models/train.py` los declara (ver `_tipos_confiables_skops`).
  `src/taxi/flows/training.py` eligió `serialization_format="cloudpickle"` para
  el mismo caso, y lo dice en un comentario. Las dos son defendibles: skops paga
  mantenimiento de la lista a cambio de seguridad; cloudpickle paga confianza en
  el origen del artefacto a cambio de que nunca falle al guardar.

Y un efecto colateral que se ve en clase: con skops, un pipeline que arrastra sin
querer un objeto de Optuna dentro del estimador **falla al loguearse**. Con
cloudpickle habría funcionado en silencio, dejando un artefacto inflado que solo
se puede cargar donde Optuna esté instalado. El fallo ruidoso es la característica,
no el defecto.

## 8. Lo mismo, ya hecho en el paquete

Todo lo de este notebook está en `src/taxi/models/train.py`, en
`_loguear_entrenamiento`: tags de datos, params, métricas globales **y por
subgrupo**, dos figuras, la firma con los tipos ensanchados y el modelo con sus
tipos confiables declarados.

```bash
uv run taxi train                     # un baseline, logueado completo
uv run taxi train --comparar          # media, lineal, random forest, xgboost
```

El notebook explora; el paquete es lo que corre en CI y en el pipeline de S04.
Si esto se implementara dos veces, en dos meses habría dos definiciones de
"valid_rmse" que no coinciden — que es literalmente lo que pasaba antes.

## Siguiente

[`03-hpo-y-registry.ipynb`](03-hpo-y-registry.ipynb): búsqueda de
hiperparámetros con runs anidados, Model Registry con aliases y la model card.

Y para el **dónde** en lugar del **qué**:
[`../scenarios/`](../scenarios/) — file store local, servidor local y AWS.